In [3]:
import pandas as pd

In [4]:
df=pd.read_csv('classification_vehicle_risk.csv')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   vehicle_age_years     4794 non-null   float64
 1   mileage_km            4784 non-null   float64
 2   engine_size_l         4811 non-null   float64
 3   service_count         5000 non-null   float64
 4   driving_style         4797 non-null   str    
 5   road_type             4796 non-null   str    
 6   fuel_type             5000 non-null   str    
 7   previous_accidents    4836 non-null   float64
 8   annual_distance_km    4810 non-null   float64
 9   maintenance_cost_usd  5000 non-null   float64
 10  risk_score            5000 non-null   float64
 11  maintenance_risk      5000 non-null   str    
dtypes: float64(8), str(4)
memory usage: 468.9 KB


In [6]:
df.isnull().sum()

vehicle_age_years       206
mileage_km              216
engine_size_l           189
service_count             0
driving_style           203
road_type               204
fuel_type                 0
previous_accidents      164
annual_distance_km      190
maintenance_cost_usd      0
risk_score                0
maintenance_risk          0
dtype: int64

In [7]:
def missval(df):
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype=='str':
                df[col]=df[col].fillna(df[col].mode()[0])
            else:
                df[col]=df[col].fillna(df[col].mean())
    return df


In [8]:
df=missval(df)
df.isnull().sum()


vehicle_age_years       0
mileage_km              0
engine_size_l           0
service_count           0
driving_style           0
road_type               0
fuel_type               0
previous_accidents      0
annual_distance_km      0
maintenance_cost_usd    0
risk_score              0
maintenance_risk        0
dtype: int64

In [9]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
def encodla(df):
    for col in df.columns:
        if df[col].dtype=='str':
            if df[col].nunique()<=3:
                dummies=pd.get_dummies(df[col],prefix=col,dtype=float)
                df=pd.concat([df.drop(columns=col),dummies],axis=1)
            else:
                df[col]=encoder.fit_transform(df[col])
    return df

In [10]:
df=encodla(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   vehicle_age_years         5000 non-null   float64
 1   mileage_km                5000 non-null   float64
 2   engine_size_l             5000 non-null   float64
 3   service_count             5000 non-null   float64
 4   fuel_type                 5000 non-null   int64  
 5   previous_accidents        5000 non-null   float64
 6   annual_distance_km        5000 non-null   float64
 7   maintenance_cost_usd      5000 non-null   float64
 8   risk_score                5000 non-null   float64
 9   maintenance_risk          5000 non-null   int64  
 10  driving_style_aggressive  5000 non-null   float64
 11  driving_style_calm        5000 non-null   float64
 12  driving_style_normal      5000 non-null   float64
 13  road_type_city            5000 non-null   float64
 14  road_type_highway  

In [11]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
num_col=df.select_dtypes(include=['int64','float64']).columns.drop('maintenance_risk')
df[num_col]=scaler.fit_transform(df[num_col])

In [12]:
df.head()

,vehicle_age_years,mileage_km,engine_size_l,service_count,fuel_type,previous_accidents,annual_distance_km,maintenance_cost_usd,risk_score,maintenance_risk,driving_style_aggressive,driving_style_calm,driving_style_normal,road_type_city,road_type_highway,road_type_mixed
0,0.25,0.289420,0.455526,0.250000,0.0,0.00,0.556245,0.301300,0.258824,1,0.0,1.0,0.0,0.0,1.0,0.0
1,0.80,0.294102,0.417790,0.083333,1.0,0.00,0.214436,0.452516,0.564706,0,0.0,1.0,0.0,1.0,0.0,0.0
2,0.65,0.195762,0.390836,0.250000,1.0,0.00,0.572644,0.337573,0.491979,2,0.0,0.0,1.0,0.0,0.0,1.0
3,0.55,0.510569,0.156334,0.333333,0.0,0.25,0.096062,0.606746,0.734759,3,0.0,0.0,1.0,0.0,0.0,1.0
4,0.15,0.237663,0.315364,0.750000,1.0,0.00,0.412065,0.304598,0.382888,1,1.0,0.0,0.0,0.0,1.0,0.0


In [13]:
from sklearn.model_selection import train_test_split
x=df.drop('maintenance_risk',axis=1)
y=df['maintenance_risk']

In [14]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [15]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression()
lr

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [16]:
lr.fit(x_train,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [17]:
y_pred=lr.predict(x_test)

In [18]:
y_pred

array([0, 0, 0, 2, 1, 0, 2, 1, 3, 1, 1, 2, 3, 0, 0, 1, 3, 3, 3, 2, 0, 3,
       3, 3, 3, 1, 2, 1, 2, 0, 0, 1, 1, 2, 2, 1, 1, 1, 1, 3, 0, 1, 0, 1,
       3, 3, 1, 0, 2, 1, 0, 1, 3, 3, 1, 2, 3, 3, 0, 3, 2, 3, 1, 3, 3, 2,
       3, 1, 3, 3, 0, 1, 3, 0, 1, 2, 0, 3, 3, 3, 3, 0, 2, 1, 1, 3, 2, 3,
       2, 2, 1, 3, 1, 1, 2, 0, 3, 2, 3, 0, 0, 0, 0, 2, 2, 2, 3, 2, 0, 2,
       1, 2, 3, 3, 0, 3, 1, 0, 2, 0, 3, 3, 2, 1, 1, 2, 2, 2, 0, 2, 3, 2,
       3, 3, 2, 0, 1, 3, 3, 0, 2, 1, 0, 2, 3, 1, 0, 3, 0, 0, 1, 3, 3, 3,
       3, 1, 2, 1, 2, 0, 1, 0, 0, 1, 2, 1, 0, 0, 3, 1, 3, 2, 0, 0, 1, 2,
       3, 3, 3, 1, 2, 1, 2, 1, 1, 0, 1, 2, 2, 2, 2, 2, 0, 2, 1, 2, 2, 1,
       1, 2, 1, 3, 2, 1, 0, 0, 2, 2, 2, 0, 1, 2, 0, 0, 0, 1, 3, 1, 1, 2,
       0, 3, 1, 0, 3, 3, 3, 3, 2, 1, 2, 1, 2, 0, 1, 3, 2, 0, 0, 0, 3, 1,
       3, 2, 3, 0, 1, 2, 2, 2, 2, 0, 2, 3, 0, 0, 3, 3, 2, 1, 0, 2, 0, 2,
       1, 0, 0, 3, 3, 3, 0, 1, 1, 0, 0, 0, 0, 0, 1, 3, 1, 2, 1, 3, 3, 3,
       1, 1, 0, 1, 3, 2, 1, 0, 3, 1, 2, 0, 1, 3, 0,

In [19]:
from sklearn.metrics import classification_report
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.79      0.77      0.78       251
           1       0.91      0.92      0.92       268
           2       0.79      0.79      0.79       250
           3       0.89      0.90      0.89       231

    accuracy                           0.85      1000
   macro avg       0.84      0.85      0.85      1000
weighted avg       0.85      0.85      0.85      1000



In [20]:
from sklearn.model_selection import KFold,cross_val_score
kf=KFold(n_splits=5,shuffle=True,random_state=42)
scores=cross_val_score(lr,x,y,cv=kf,scoring='accuracy')


c:\Users\Fazliddin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Fazliddin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https

In [21]:
print(scores)

[0.846 0.846 0.85  0.867 0.852]


In [22]:
import numpy as np
np.mean(scores)

np.float64(0.8522000000000001)

In [23]:
from sklearn.tree import DecisionTreeClassifier
dtc=DecisionTreeClassifier()
dtc

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [24]:
dtc.fit(x_train,y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [25]:
y_pred=dtc.predict(x_test)
y_pred

array([0, 2, 0, 0, 1, 0, 2, 1, 0, 1, 1, 2, 3, 0, 0, 2, 3, 3, 3, 2, 0, 3,
       3, 3, 3, 1, 0, 1, 0, 0, 0, 1, 1, 2, 2, 1, 1, 1, 1, 3, 0, 1, 0, 2,
       0, 3, 2, 0, 2, 1, 0, 1, 3, 3, 1, 2, 3, 3, 0, 3, 2, 3, 1, 3, 3, 2,
       3, 2, 3, 3, 0, 1, 3, 0, 1, 2, 2, 3, 3, 3, 3, 0, 2, 1, 1, 3, 2, 3,
       2, 2, 1, 0, 1, 1, 2, 3, 3, 2, 3, 0, 0, 0, 0, 1, 2, 2, 3, 2, 0, 2,
       1, 0, 0, 0, 0, 3, 1, 2, 0, 0, 3, 3, 0, 1, 1, 2, 2, 1, 3, 2, 3, 2,
       3, 0, 2, 0, 1, 3, 3, 0, 0, 1, 2, 2, 3, 1, 0, 3, 0, 0, 1, 0, 3, 0,
       3, 1, 2, 1, 2, 0, 1, 0, 0, 1, 2, 1, 0, 0, 3, 1, 0, 2, 0, 0, 1, 0,
       3, 3, 3, 1, 2, 2, 2, 1, 2, 0, 1, 2, 2, 2, 2, 2, 0, 2, 1, 2, 2, 1,
       2, 2, 1, 3, 2, 1, 0, 0, 2, 2, 2, 0, 1, 0, 0, 0, 0, 2, 3, 1, 1, 2,
       2, 3, 2, 0, 3, 3, 3, 3, 2, 1, 2, 1, 2, 0, 1, 3, 2, 0, 2, 0, 3, 1,
       3, 2, 3, 0, 1, 2, 2, 2, 1, 2, 2, 3, 0, 3, 3, 3, 2, 1, 0, 2, 0, 2,
       1, 0, 0, 0, 0, 3, 0, 1, 2, 0, 0, 0, 3, 0, 1, 3, 1, 2, 1, 3, 3, 3,
       1, 2, 0, 2, 3, 1, 1, 3, 0, 1, 2, 0, 1, 3, 0,

In [26]:
from sklearn.metrics import classification_report
report=classification_report(y_test,y_pred)
print(report)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       251
           1       1.00      1.00      1.00       268
           2       0.99      0.99      0.99       250
           3       1.00      1.00      1.00       231

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [27]:
from sklearn.model_selection import KFold,cross_val_score
kf=KFold(n_splits=5,shuffle=True,random_state=42)
scores=cross_val_score(dtc,x,y,cv=kf,scoring='accuracy')
print(scores)

[0.996 0.997 0.998 0.998 0.994]


In [28]:
np.mean(scores)

np.float64(0.9965999999999999)

In [29]:
from sklearn.ensemble import RandomForestClassifier
rfc=RandomForestClassifier()
rfc

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [30]:
rfc.fit(x_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [31]:
y_pred=rfc.predict(x_test)
print(y_pred)

[0 2 0 0 1 0 2 1 0 1 1 2 3 0 0 2 3 3 3 2 0 3 3 3 3 1 0 1 0 0 0 1 1 2 2 1 1
 1 1 3 0 1 0 2 0 3 2 0 2 1 0 1 3 3 1 2 3 3 0 3 2 3 1 3 3 2 3 2 3 3 0 1 3 0
 1 2 0 3 3 3 3 0 2 1 1 3 2 3 2 2 1 0 1 1 2 3 3 2 3 0 0 0 0 1 2 2 3 2 0 2 1
 0 0 0 0 3 1 2 0 0 3 3 0 1 1 2 2 1 3 2 3 2 3 0 2 0 1 3 3 0 0 1 2 2 3 1 0 3
 0 0 1 0 3 0 3 1 2 1 2 0 1 0 0 1 2 1 0 0 3 1 0 2 0 0 1 0 3 3 3 1 2 2 2 1 2
 0 1 2 2 2 2 2 0 2 1 2 2 1 2 2 1 3 2 1 0 0 2 2 2 0 1 0 0 0 0 2 3 1 1 2 2 3
 2 0 3 3 3 3 2 1 2 1 2 0 1 3 2 0 2 0 3 1 3 2 3 0 1 2 2 2 1 2 2 3 0 3 3 3 2
 1 0 2 0 2 1 0 0 0 0 3 0 1 2 0 0 2 3 0 1 3 1 2 1 3 3 3 1 2 0 2 3 1 1 3 0 1
 2 0 1 3 0 2 2 3 2 0 0 3 1 3 1 0 2 2 0 0 1 1 1 3 1 1 0 0 1 0 2 2 0 2 0 0 2
 0 2 2 1 3 0 2 3 0 0 3 3 0 3 3 3 0 1 3 3 1 1 0 0 0 2 0 2 2 1 3 3 3 2 1 1 3
 0 3 3 0 0 2 3 0 3 3 1 1 1 2 1 2 1 2 0 1 0 2 1 2 3 1 3 2 0 0 2 2 3 1 3 1 3
 3 1 1 2 2 0 0 3 1 3 3 0 3 1 1 3 3 1 3 0 0 1 2 1 2 3 1 1 2 0 0 2 1 1 1 3 3
 0 1 0 0 3 0 0 3 1 3 2 0 2 0 0 0 2 2 3 3 1 1 3 3 3 0 1 0 3 0 2 3 1 0 0 2 1
 2 3 0 1 2 1 2 0 3 2 3 1 

In [32]:
from sklearn.metrics import classification_report
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       251
           1       1.00      0.99      1.00       268
           2       0.99      1.00      0.99       250
           3       1.00      1.00      1.00       231

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [33]:
from sklearn.model_selection import KFold,cross_val_score
kf=KFold(n_splits=5,shuffle=True,random_state=42)
scores=cross_val_score(rfc,x,y,scoring='accuracy')
print(scores)

[0.997 0.995 0.994 0.997 0.996]


In [34]:
np.mean(scores)

np.float64(0.9957999999999998)

In [35]:
from sklearn.svm import SVC
svc=SVC(kernel='linear',C=1.0)


In [36]:
svc.fit(x_train,y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [37]:
y_pred=svc.predict(x_test)
y_pred

array([0, 0, 0, 0, 1, 0, 2, 1, 3, 1, 1, 2, 3, 0, 0, 1, 3, 3, 3, 2, 0, 3,
       3, 3, 3, 1, 0, 1, 0, 0, 0, 1, 1, 2, 2, 1, 1, 1, 1, 3, 0, 1, 0, 2,
       0, 3, 1, 0, 2, 1, 0, 1, 3, 3, 1, 2, 3, 3, 0, 3, 2, 3, 1, 3, 3, 2,
       3, 2, 3, 3, 0, 1, 3, 0, 1, 2, 0, 3, 3, 3, 3, 0, 2, 1, 1, 3, 2, 3,
       2, 2, 1, 3, 1, 1, 2, 3, 3, 2, 3, 0, 0, 0, 0, 1, 2, 2, 3, 2, 0, 2,
       1, 0, 0, 3, 0, 3, 1, 0, 2, 0, 3, 3, 0, 1, 1, 2, 2, 1, 0, 2, 3, 2,
       3, 3, 2, 0, 1, 3, 3, 0, 0, 1, 2, 2, 3, 1, 0, 3, 0, 0, 1, 0, 3, 0,
       3, 1, 2, 1, 2, 0, 1, 0, 0, 1, 2, 1, 0, 0, 3, 1, 3, 2, 0, 0, 1, 0,
       3, 3, 3, 1, 2, 1, 2, 1, 2, 0, 1, 2, 2, 2, 2, 2, 0, 2, 1, 2, 2, 1,
       2, 2, 1, 3, 2, 1, 0, 0, 2, 2, 2, 0, 1, 0, 0, 0, 0, 2, 3, 1, 1, 2,
       2, 3, 1, 0, 3, 3, 3, 3, 2, 1, 2, 1, 2, 0, 1, 3, 2, 0, 2, 0, 3, 1,
       3, 2, 3, 0, 1, 2, 2, 2, 2, 2, 2, 3, 0, 0, 3, 3, 2, 1, 0, 2, 0, 2,
       1, 0, 0, 3, 0, 3, 0, 1, 2, 0, 0, 0, 0, 0, 1, 3, 1, 2, 1, 3, 3, 3,
       1, 2, 0, 2, 3, 2, 1, 0, 3, 1, 2, 0, 1, 3, 0,

In [38]:
from sklearn.metrics import accuracy_score
scores=accuracy_score(y_pred,y_test)
print(scores)

0.937


In [39]:
from sklearn.model_selection import KFold,cross_val_score
kf=KFold(n_splits=5,shuffle=True,random_state=42)
scores=cross_val_score(svc,x,y,cv=kf,scoring="accuracy")
print(scores)

[0.937 0.922 0.938 0.935 0.94 ]


In [40]:
np.mean(scores)

np.float64(0.9343999999999999)

In [41]:
from tabulate import tabulate
results=[
    ['LogisticRegression',0.852],
    ['DecisionTreeClassifier',0.99],
    ['RandomForestClassifier',0.996],
    ['SVC',0.934]
]
headers=['Algorithm','Accuracy score']
table=tabulate(results,headers=headers,tablefmt='grid',floatfmt='.2f')
print(table)

+------------------------+------------------+
| Algorithm              |   Accuracy score |
+========================+==================+
| LogisticRegression     |             0.85 |
+------------------------+------------------+
| DecisionTreeClassifier |             0.99 |
+------------------------+------------------+
| RandomForestClassifier |             1.00 |
+------------------------+------------------+
| SVC                    |             0.93 |
+------------------------+------------------+


In [42]:
import os

In [43]:
output_folder='model/trained'   
os.makedirs(output_folder,exist_ok=True)
output_path=os.path.join(output_folder,'trained_model.csv')
df.to_csv(output_path,index=False) 
print(f"Trained model saved to : {output_path}")

Trained model saved to : model/trained\trained_model.csv


yangi fayl paydo bo'ldi